In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

os.chdir("../RecSys_Course_AT_PoliMi")

!pwd

#!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/RecSys_Challenge_2025-26/RecSys_Course_AT_PoliMi


In [3]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

Tensorflow is not available


In [4]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [5]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [6]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [ ]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2] # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [ ]:

import time 
import gc

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):
                          
    start_time = time.time()
    scores = []
    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        
        
        #Cambiare il modello qui sotto, insieme al range e ai parametri 
        
        
        recommender_instance = SLIMElasticNetRecommender(URM_combined)
        recommender_instance.fit(topK = optuna_trial.suggest_int("topK", 50, 1000),
                             l1_ratio = optuna_trial.suggest_float("l1_ratio", 1e-5, 1.0, log=True),
                             alpha = optuna_trial.suggest_float("alpha", 1e-5, 1e-2, log=True)
                             )
        
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        #print("prova = ", result["MAP"].values[0])
        #print(result)
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break

        del recommender_instance
        del URM_combined
        del evaluator_test
        gc.collect() 
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)

In [9]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
timeout_seconds = 16 * 60 * 60 
        
save_results = SaveResults()
        

optuna_study.optimize(objective_function_funksvd,
    callbacks=[save_results],
    n_trials=500,           # Aumentato il numero di trial massimi
    timeout=timeout_seconds, # Il processo si ferma allo scadere delle 16 ore
    n_jobs=1                # Mantieni 1 per sicurezza sulla RAM
)

[I 2025-12-31 18:18:54,134] A new study created in memory with name: no-name-10b51c49-98a8-4693-bdb9-79448811c3e2


SLIMElasticNetRecommender: Processed 4822 (69.2%) in 5.00 min. Items per second: 16.07
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.22 min. Items per second: 16.08
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 7.06 sec. Users per second: 3831
SLIMElasticNetRecommender: Processed 4848 (69.6%) in 5.00 min. Items per second: 16.15
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.17 min. Items per second: 16.19
EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27058 (100.0%) in 7.11 sec. Users per second: 3807
SLIMElasticNetRecommender: Processed 4857 (69.7%) in 5.00 min. Items per second: 16.18
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.17 min. Items per second: 16.20
EvaluatorHoldout: Ignoring 32 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27063 (100.0%) in 7.03 sec. Users per

[I 2025-12-31 18:55:19,443] Trial 0 finished with value: 0.2870734463583884 and parameters: {'topK': 702, 'l1_ratio': 0.3269595896798567, 'alpha': 0.0006442842722906306}. Best is trial 0 with value: 0.2870734463583884.


[0.28621945503686413, 0.2856186510129422, 0.28701921082554505, 0.2887906272774206, 0.2877192876391699]
SLIMElasticNetRecommender: Processed 3883 (55.7%) in 5.00 min. Items per second: 12.94
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 8.92 min. Items per second: 13.01
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 5.34 sec. Users per second: 5063
SLIMElasticNetRecommender: Processed 3890 (55.8%) in 5.00 min. Items per second: 12.96
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 8.98 min. Items per second: 12.93
EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27058 (100.0%) in 5.36 sec. Users per second: 5044
SLIMElasticNetRecommender: Processed 3874 (55.6%) in 5.00 min. Items per second: 12.91
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 8.95 min. Items per second: 12.97
EvaluatorHoldout: Ignoring 32 ( 0.1%) Users th

[I 2025-12-31 19:40:32,988] Trial 1 finished with value: 0.2803888868837202 and parameters: {'topK': 207, 'l1_ratio': 0.008024209358438455, 'alpha': 0.00021017040127447085}. Best is trial 0 with value: 0.2870734463583884.


...

[0.2863124732162092, 0.28526331920397946, 0.2868960459465752, 0.2882083433311354, 0.2875048982860903]
SLIMElasticNetRecommender: Processed 3994 (57.3%) in 5.00 min. Items per second: 13.31
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 8.71 min. Items per second: 13.33
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 6.32 sec. Users per second: 4285
SLIMElasticNetRecommender: Processed 4008 (57.5%) in 5.00 min. Items per second: 13.36
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 8.69 min. Items per second: 13.37
EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27058 (100.0%) in 6.31 sec. Users per second: 4291
SLIMElasticNetRecommender: Processed 3982 (57.1%) in 5.00 min. Items per second: 13.27
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 8.71 min. Items per second: 13.33
EvaluatorHoldout: Ignoring 32 ( 0.1%) Users tha

[I 2026-01-01 09:47:35,807] Trial 11 finished with value: 0.28740744282426267 and parameters: {'topK': 980, 'l1_ratio': 0.0008199075676765447, 'alpha': 0.0015330846977827794}. Best is trial 5 with value: 0.287785682823623.


[0.28672187546053046, 0.2857706571680841, 0.28728983462437624, 0.2891846588731932, 0.2880701879951295]
SLIMElasticNetRecommender: Processed 4673 (67.1%) in 5.00 min. Items per second: 15.57
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.45 min. Items per second: 15.59
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 6.40 sec. Users per second: 4226
SLIMElasticNetRecommender: Processed 4678 (67.1%) in 5.00 min. Items per second: 15.59
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.45 min. Items per second: 15.59
EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27058 (100.0%) in 6.39 sec. Users per second: 4235
SLIMElasticNetRecommender: Processed 4672 (67.0%) in 5.00 min. Items per second: 15.57
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.46 min. Items per second: 15.58
EvaluatorHoldout: Ignoring 32 ( 0.1%) Users th

[I 2026-01-01 10:25:22,558] Trial 12 finished with value: 0.28756079452295125 and parameters: {'topK': 958, 'l1_ratio': 0.02061603569038766, 'alpha': 0.0014247921001518614}. Best is trial 5 with value: 0.287785682823623.


[0.2869256336794301, 0.28599002129326695, 0.28736760982260456, 0.28915464995992396, 0.2883660578595307]


In [10]:
optuna_study.best_trial.params

{'topK': 564, 'l1_ratio': 0.009383806291988516, 'alpha': 0.002145871959946555}

In [11]:
save_results.results_df

,result,train_time (min)
0,0.287073,36.421765
1,0.280389,45.225674
2,0.282829,309.648369
3,0.283777,37.480223
4,0.287672,40.179569
5,0.287786,37.845425
6,0.283407,126.650581
7,0.283609,51.228569
8,0.282777,45.254648
9,0.277379,123.157072


In [12]:
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
best_hyperparams

{}